# Unit Testing

**Seong Cho** · ACME Python Essentials, Lab 7

**Lab objective:** learn to write unit tests in Python with `pytest` and practice test-driven development, a formal strategy for finding and eliminating errors quickly as a program is built and for making sure it still works whenever it is modified.

<sub>Lab specification from the *Python Essentials* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

**How this notebook is organized.** The lab is about writing a *separate* test file, so the code lives in two real files next to this notebook: [`specs.py`](specs.py) (the functions under test) and [`test_specs.py`](test_specs.py) (the `pytest` unit tests). Those files are the source of truth. The cells below import them and display the relevant source, then run `pytest` on them.

In [1]:
import sys
import inspect
import pytest
from IPython.display import display

import specs
import test_specs

def show(*objects):
    """Display the source code of functions or classes from specs.py / test_specs.py."""
    src = "\n\n".join(inspect.getsource(obj).rstrip() for obj in objects)
    display({"text/markdown": "```python\n" + src + "\n```", "text/plain": src}, raw=True)

## Unit Tests

A *unit test* checks a small piece of code (usually a function or class method) for correctness, independent of the rest of the program, by running test cases and comparing actual outputs with expected outputs. Each test case is usually an `assert` statement: a shortcut for raising an `AssertionError`, with an optional message, if a boolean statement is false.

In [2]:
result = str(5) == '5'
if result is False:                     # The long way...
    raise AssertionError("incorrect result")
assert result, "incorrect result"       # ...and the same check in one line.

try:
    assert 5 == '5', "5 is not a string"    # Asserting a false statement raises an AssertionError.
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: 5 is not a string


To test a simple `add()` function in `specs.py`, a corresponding file called `test_specs.py`, which holds all of the unit tests for `specs.py`, contains a unit test called `test_add()`. Unit test functions don't need to return anything, but they should raise an exception if a test case fails.

In [3]:
show(specs.add, test_specs.test_add)
test_specs.test_add()       # Passes silently.

```python
def add(a, b):
    """Add two numbers."""
    return a + b

def test_add():
    assert specs.add(1, 3) == 4, "failed on positive integers"
    assert specs.add(-5, -7) == -12, "failed on negative integers"
    assert specs.add(-6, 14) == 8
```

> **Note.** This style of external testing (checking that certain inputs give certain outputs) is called *black box testing*. The code's structure isn't considered, only what it produces. The author of a black box test doesn't even need to be the person who writes the program; having one person write tests and another write the code helps catch problems that either might miss alone.

## PyTest

Python's `pytest` module provides tools for building tests, running them, and reporting detailed results. It isn't part of the standard library but is included in Anaconda (`pip install -U pytest` if needed); the standard library's `unittest` also provides a testing framework, but is less popular and straightforward. Run `py.test` in a directory to collect and run its tests. The lab's sample output, as written:

```text
$ py.test
============================= test session starts =============================
platform darwin -- Python 3.6.0, pytest-3.0.5, py-1.4.32, pluggy-0.4.0
rootdir: /Users/Student/example_tests, inifile:
collected 12 items

test_calendar.py ........
test_google.py .F..

================================== FAILURES ===================================
________________________________ test_subtract ________________________________

    def test_subtract():
>       assert google.subtract(42, 17)==25, "subtract() failed for a > b > 0"
E       AssertionError: subtract() failed for a > b > 0
E       assert 35 == 25
E        +  where 35 = <function subtract at 0x102d4eb90>(42, 17)
E        +    where <function subtract at 0x102d4eb90> = google.subtract

test_google.py:11: AssertionError
===================== 1 failed, 11 passed in 0.02 seconds =====================
```

Each `.` is a passed test and each `F` a failed one, in order, so only the second of four tests in `test_google.py` failed.

In this notebook the suite is run with `!{sys.executable} -m pytest ...` (the same `py.test` run, using the kernel's Python). `-k <name>` selects tests whose names match, and `-p no:cacheprovider` just stops `pytest` from leaving a `.pytest_cache` folder behind.

In [4]:
!{sys.executable} -m pytest -q -p no:cacheprovider test_specs.py -k "add or divide"

...                                                                      [100%]
3 passed, 12 deselected in 0.01s


> **Achtung!** PyTest will not find or run tests if they are not contained in files named `test_*.py` or `*_test.py`, where `*` represents any number of characters. In addition, the unit tests themselves must be named `test_*()` or `*_test()`. To change this behavior, see http://pytest.org/latest/example/pythoncollection.html.

## Problem 1: Find the bug in `smallest_factor()`

> **Problem 1.** The following function contains a subtle but important error.
>
> ```python
> def smallest_factor(n):
>     """Return the smallest prime factor of the positive integer n."""
>     if n == 1: return 1
>     for i in range(2, int(n**.5)):
>         if n % i == 0: return i
>     return n
> ```
>
> Write a unit test for this function, including test cases that you suspect might uncover the error (what are the edge cases for this function?). Use `pytest` to run your unit test and discover a test case that fails, then use this information to correct the function.

The edge cases are perfect squares of primes: `range(2, int(n**.5))` stops one short of $\sqrt{n}$, so for $n = 4$ (or $169 = 13^2$) the loop never tries the actual factor and the function returns $n$ itself. The test:

In [5]:
show(test_specs.test_smallest_factor)

```python
def test_smallest_factor():
    assert specs.smallest_factor(1) == 1, "failed on base case n = 1"
    assert specs.smallest_factor(2) == 2, "failed on base case n = 2"
    assert specs.smallest_factor(3) == 3, "failed on base case n = 3"
    assert specs.smallest_factor(4) == 2, "failed on composition n = 4 = 2*2"
    assert specs.smallest_factor(5) == 5, "failed on base case n = 5"
    assert specs.smallest_factor(6) == 2, "failed on composition n = 6 = 3*2"
    assert specs.smallest_factor(10) == 2, "failed on composition n = 10 = 5*2"
    assert specs.smallest_factor(15) == 3, "failed on composition n = 15 = 5*3"
    assert specs.smallest_factor(99) == 3, "failed on large number n = 99"
    assert specs.smallest_factor(169) == 13, "failed on perfect square n = 169 = 13**2"
    assert specs.smallest_factor(1073) == 29, "failed on prime composition n = 1073 = 29*37"
```

Running those test cases against the function as given in the spec fails on the first perfect square:

In [6]:
def smallest_factor_as_given(n):
    """The spec's original (buggy) version."""
    if n == 1: return 1
    for i in range(2, int(n**.5)):
        if n % i == 0: return i
    return n

print(smallest_factor_as_given(4), smallest_factor_as_given(169))    # Should be 2 and 13.

4 169


The fix in `specs.py` extends the range to `int(n**.5) + 1`, and the test now passes:

In [7]:
show(specs.smallest_factor)
!{sys.executable} -m pytest -q -p no:cacheprovider test_specs.py -k smallest_factor

```python
def smallest_factor(n):
    """Return the smallest prime factor of the positive integer n."""
    if n == 1: return 1
    for i in range(2, int(n**.5) + 1):
        if n % i == 0: return i
    return n
```

.                                                                        [100%]
1 passed, 14 deselected in 0.01s


## Coverage

Successful unit tests include enough test cases to test the entire program. *Coverage* refers to the number of lines of code that are executed by at least one test case. One tool for measuring it is `pytest-cov`, an extension of `pytest` that must be installed separately (`conda install pytest-cov`). The flag `--cov` adds coverage information to the `py.test` output. The lab's sample output, as written:

```text
$ py.test --cov
============================= test session starts =============================
platform darwin -- Python 3.6.0, pytest-3.0.5, py-1.4.32, pluggy-0.4.0
rootdir: /Users/Student/Testing, inifile:
plugins: cov-2.3.1
collected 7 items

test_specs.py .......

---------- coverage: platform darwin, python 3.6.0-final-0 ----------
Name            Stmts   Miss  Cover
-----------------------------------
specs.py           73     34    53%
test_specs.py      46      0   100%
-----------------------------------
TOTAL             119     34    71%

========================== 7 passed in 0.03 seconds ===========================
```

`Stmts` is the number of lines of code covered by a unit test and `Miss` the number not yet covered. Test files generally have 100% coverage, since `pytest` runs them in their entirety. To see *which* lines are missed, `py.test --cov-report html --cov` writes an HTML report to `htmlcov/`; `htmlcov/specs_py.html` highlights the uncovered lines of `specs.py` in red.

> **Note.** Statement coverage is *white box testing*, because it requires an understanding of the code's structure. Most black box tests can be written before a program is implemented, but white box tests should be added after the program is completed. Designing tests to cover every statement may reveal unreachable lines, a conditional that isn't working as intended, or problems with edge cases.

The cell below prints a terminal coverage report (`term-missing` lists the line numbers of any missed lines) when `pytest-cov` is installed.

In [8]:
import importlib.util

if importlib.util.find_spec("pytest_cov"):
    !{sys.executable} -m pytest -q -p no:cacheprovider --cov=specs --cov-report term-missing test_specs.py
else:
    print("pytest-cov is not installed in this environment, so the coverage report is skipped.")

pytest-cov is not installed in this environment, so the coverage report is skipped.


## Problem 2: Testing `month_length()`

> **Problem 2.** With `pytest-cov` installed, check your coverage of `smallest_factor()` from Problem 1. Write additional test cases if necessary to get complete coverage. Then, write a comprehensive unit test for the following (correctly written) function.
>
> ```python
> def month_length(month, leap_year=False):
>     """Return the number of days in the given month."""
>     if month in {"September", "April", "June", "November"}:
>         return 30
>     elif month in {"January", "March", "May", "July",
>                         "August", "October", "December"}:
>         return 31
>     if month == "February":
>         if not leap_year:
>             return 28
>         else:
>             return 29
>     else:
>         return None
> ```

The test hits every branch: a 30-day month, a 31-day month, February with and without a leap year, and an invalid input.

In [9]:
show(test_specs.test_month_length)
!{sys.executable} -m pytest -q -p no:cacheprovider test_specs.py -k month_length

```python
def test_month_length():
    assert specs.month_length(5) == None, "failed for invalid input"
    assert specs.month_length("April") == 30, "failed on April for set of months with 30 days"
    assert specs.month_length("August") == 31, "failed on August for set of months with 31 days"
    assert specs.month_length("February") == 28, "failed on February for non leap year"
    assert specs.month_length("February", leap_year=True) == 29, "failed on February for leap year"
```

.                                                                        [100%]
1 passed, 14 deselected in 0.02s


## Testing Exceptions

A good unit test checks that a program raises the exceptions it is supposed to raise, and doesn't raise any unexpected ones. `pytest.raises()` asserts that a call raises a given exception: `pytest.raises(ZeroDivisionError, specs.divide, a=4, b=0)` passes only if that call raises a `ZeroDivisionError` (a different exception, or none, fails the test). Used with a `with` statement, it also captures the exception in `excinfo` so the test can check that the error came from the intended `raise` statement: the exception object is `excinfo.value`, and `excinfo.value.args[0]` is its message.

In [10]:
show(specs.divide, test_specs.test_divide)

with pytest.raises(ZeroDivisionError) as excinfo:
    specs.divide(4, 0)
excinfo.value.args[0]

```python
def divide(a, b):
    """Divide two numbers, raising an error if the second number is zero."""
    if b == 0:
        raise ZeroDivisionError("second input cannot be zero")
    return a / b

def test_divide():
    assert specs.divide(4,2) == 2, "integer division"
    assert specs.divide(5,4) == 1.25, "float division"
    with pytest.raises(ZeroDivisionError) as excinfo:
        specs.divide(4, 0)
    assert excinfo.value.args[0] == "second input cannot be zero"
```

'second input cannot be zero'

## Problem 3: Testing `operate()`

> **Problem 3.** Write a comprehensive unit test for the following function. Make sure that each exception is raised properly by explicitly checking the exception message. Use `pytest-cov` and its `cov-report` tool to confirm that you have full coverage for this function.
>
> ```python
> def operate(a, b, oper):
>     """Apply an arithmetic operation to a and b."""
>     if type(oper) is not str:
>         raise TypeError("oper must be a string")
>     elif oper == '+':
>         return a + b
>     elif oper == '-':
>         return a - b
>     elif oper == '*':
>         return a * b
>     elif oper == '/':
>         if b == 0:
>             raise ZeroDivisionError("division by zero is undefined")
>         return a / b
>     raise ValueError("oper must be one of '+', '/', '-', or '*'")
> ```

Each of the four operators is checked, and each of the three exceptions is triggered and its message compared.

In [11]:
show(test_specs.test_operate)
!{sys.executable} -m pytest -q -p no:cacheprovider test_specs.py -k operate

```python
def test_operate():
    assert specs.operate(4, 5, '+') == 9, "failed for '+' operator"
    assert specs.operate(3, 6, '-') == -3, "failed for '-' operator"
    assert specs.operate(6, -7, '*') == -42, "failed for '*' operator"
    assert specs.operate(15, 3, '/') == 5, "failed for '/' operator"
    with pytest.raises(TypeError) as excinfo_1:
        specs.operate(14, 2, 5)
    assert excinfo_1.value.args[0] == "oper must be a string"
    with pytest.raises(ZeroDivisionError) as excinfo_2:
        specs.operate(4, 0, '/')
    assert excinfo_2.value.args[0] == "division by zero is undefined"
    with pytest.raises(ValueError) as excinfo_3:
        specs.operate(3, 2, '=')
    assert excinfo_3.value.args[0] == "oper must be one of '+', '/', '-', or '*'"
```

.                                                                        [100%]
1 passed, 14 deselected in 0.01s


## Fixtures

The lab provides a `Fraction` class representing rational numbers as reduced fractions, with an `__init__()` that rejects a zero denominator or non-integer inputs and reduces by the greatest common divisor, plus `__str__()`, `__float__()`, `__eq__()`, and the arithmetic magic methods `__add__()`, `__sub__()`, `__mul__()`, and `__truediv__()`.

In [12]:
from specs import Fraction

print(Fraction(8, 12))                      # 8/12 reduces to 2/3.
print(Fraction(1, 5) == Fraction(3, 15))    # 3/15 reduces to 1/5.
print(Fraction(1, 3) * Fraction(1, 4))

2/3
True
1/12


To test a class like this, it helps to have ready-made objects for every unit test. A *fixture*, a function marked with the `@pytest.fixture` decorator, sets up variables that can be used as mock data for multiple unit tests. Each test takes the fixture function as an input and unpacks the objects it built. The spec provides the fixture below and the tests `test_fraction_init()`, `test_fraction_str()`, `test_fraction_float()`, and `test_fraction_eq()` as a starting point.

In [13]:
show(test_specs.set_up_fractions)

```python
@pytest.fixture
def set_up_fractions():
    frac_1_3 = specs.Fraction(1, 3)
    frac_1_2 = specs.Fraction(1, 2)
    frac_n2_3 = specs.Fraction(-2, 3)
    return frac_1_3, frac_1_2, frac_n2_3
```

## Problem 4: Testing and fixing `Fraction`

> **Problem 4.** Add test cases to the unit tests provided above to get full coverage for the `__init__()`, `__str__()`, `__float__()`, and `__eq__()` methods. You may modify the fixture function if it helps. Also add unit tests for the magic methods `__add__()`, `__sub__()`, `__mul__()`, and `__truediv__()`. Verify that you have full coverage with `pytest-cov`.
>
> Additionally, *two* of the `Fraction` class's methods are implemented incorrectly. Use your tests to find the issues, then correct the methods so that your tests pass.
>
> See http://doc.pytest.org/en/latest/index.html for complete documentation on `pytest`.

The extended tests add the exception cases of `__init__()`, a whole-number `__str__()`, comparison against a float in `__eq__()`, and tests for the four arithmetic methods:

In [14]:
show(test_specs.test_fraction_init, test_specs.test_fraction_str, test_specs.test_fraction_float,
     test_specs.test_fraction_eq, test_specs.test_fraction_add, test_specs.test_fraction_sub,
     test_specs.test_fraction_mul, test_specs.test_fraction_truediv)

```python
def test_fraction_init(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_3.numer == 1
    assert frac_1_2.denom == 2
    assert frac_n2_3.numer == -2
    frac = specs.Fraction(30, 42)
    assert frac.numer == 5
    assert frac.denom == 7
    with pytest.raises(ZeroDivisionError) as excinfo_1:
        specs.Fraction(1, 0)
    assert excinfo_1.value.args[0] == "denominator cannot be zero"
    with pytest.raises(TypeError) as excinfo_1:
        specs.Fraction('3', '5')
    assert excinfo_1.value.args[0] == "numerator and denominator must be integers"

def test_fraction_str(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert str(frac_1_3) == "1/3"
    assert str(frac_1_2) == "1/2"
    assert str(frac_n2_3) == "-2/3"
    frac = specs.Fraction(30, 5)
    assert str(frac) == '6'

def test_fraction_float(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert float(frac_1_3) == 1 / 3.
    assert float(frac_1_2) == .5
    assert float(frac_n2_3) == -2 / 3.

def test_fraction_eq(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_2 == specs.Fraction(1, 2)
    assert frac_1_3 == specs.Fraction(2, 6)
    assert frac_n2_3 == specs.Fraction(8, -12)
    assert specs.Fraction(3, 5) == 0.6

def test_fraction_add(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_2 + frac_1_3 == specs.Fraction(5, 6)
    assert frac_1_3 + frac_n2_3 == specs.Fraction(-1, 3)

def test_fraction_sub(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_2 - frac_1_3 == specs.Fraction(1, 6)
    assert frac_1_3 - frac_n2_3 == specs.Fraction(1, 1)

def test_fraction_mul(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_2 * frac_1_3 == specs.Fraction(1, 6)
    assert frac_1_3 * frac_n2_3 == specs.Fraction(-2, 9)

def test_fraction_truediv(set_up_fractions):
    frac_1_3, frac_1_2, frac_n2_3 = set_up_fractions
    assert frac_1_2 / frac_1_3 == specs.Fraction(3, 2)
    assert frac_1_3 / frac_n2_3 == specs.Fraction(-1, 2)
    with pytest.raises(ZeroDivisionError) as excinfo:
        frac_1_3 / specs.Fraction(0, 3)
    assert excinfo.value.args[0] == "cannot divide by zero"
```

The tests exposed the two broken methods. As given in the spec, addition and subtraction multiplied numerator by numerator and denominator by denominator:

```python
def __add__(self, other):
    return Fraction(self.numer*other.numer + self.denom*other.denom,
                                                    self.denom*other.denom)
def __sub__(self, other):
    return Fraction(self.numer*other.numer - self.denom*other.denom,
                                                    self.denom*other.denom)
```

The corrected versions cross-multiply, $\frac{a}{b} \pm \frac{c}{d} = \frac{ad \pm bc}{bd}$. The spec's `__str__()` also formatted fractions as `"{} / {}"` (with spaces), which disagrees with the spec's own expected output `"1/3"`, so it was changed to `"{}/{}"`.

In [15]:
show(specs.Fraction.__str__, specs.Fraction.__add__, specs.Fraction.__sub__)
!{sys.executable} -m pytest -q -p no:cacheprovider test_specs.py -k fraction

```python
    def __str__(self):
        if self.denom != 1:
            return "{}/{}".format(self.numer, self.denom)
        else:
            return str(self.numer)

    def __add__(self, other):
        return Fraction(self.numer*other.denom + self.denom*other.numer,
                                                        self.denom*other.denom)

    def __sub__(self, other):
        return Fraction(self.numer*other.denom - self.denom*other.numer,
                                                        self.denom*other.denom)
```

........                                                                 [100%]
8 passed, 7 deselected in 0.02s


## Test-driven Development

*Test-driven development* (TDD) is the programming style of writing tests *before* implementing the actual code. It may sound tedious, but TDD encourages simple design and implementation, speeds up the actual coding, and gives measurable checkpoints for the development process:

1. Define the program specifications in great detail: function declarations, class definitions, and (especially) docstrings that say exactly what each function or method accepts and returns.
2. Write a unit test for each unit of the program (usually black box tests).
3. Implement the program code, making changes until all tests pass.

Adding features or cleaning up existing code follows the same loop (specifications → tests → implementation): redefine the specifications, add or modify tests to match, then change the code until all tests pass. If the tests are thorough enough, a program that passes them all can be considered complete, but it isn't enough just to *have* tests; they must test the code accurately and rigorously, so check the coverage and add tests if needed.

See https://en.wikipedia.org/wiki/Test-driven_development for more discussion on TDD and https://en.wikipedia.org/wiki/Behavior-driven_development for an overview of Behavior-driven development (BDD), a close relative of TDD.

## Problem 5: Tests for Set (written first)

> **Problem 5.** *Set* is a card game about finding patterns. Each card contains a design with 4 different properties: color (red, green or purple), shape (diamond, oval or squiggly), quantity (one, two, or three) and pattern (solid, striped or outlined). A *set* is a group of three cards which are either all the same or all different for each property. You can try playing Set online at http://smart-games.org/en/set/start.
>
> Here is a group of twelve Set cards. *[Image of the twelve cards not reproduced.]*
>
> This collection of cards contains six unique sets:
>
> - (a) Same in quantity and shape; different in pattern and color
> - (b) Same in color and pattern; different in shape and quantity
> - (c) Same in pattern; different in shape, quantity and color
> - (d) Same in shape; different in quantity, pattern and color
> - (e) Different in all aspects
> - (f) Different in all aspects
>
> Each Set card can be uniquely represented by a 4-bit integer in base 3,¹ where each digit represents a different property and each property has three possible values. A full hand in Set is a group of twelve unique cards, so a hand can be represented by a list of twelve 4-digit integers in base 3. For example, the hand shown above could be represented by the following list.
>
> ```python
> hand1 = ["1022", "1122", "0100", "2021",
>          "0010", "2201", "2111", "0020",
>          "1102", "0210", "2110", "1020"]
> ```
>
> The following function definitions provide a framework for partially implementing Set by calculating the number of sets in a given hand.
>
> ```python
> def count_sets(cards):
>     """Return the number of sets in the provided Set hand.
>
>     Parameters:
>         cards (list(str)) a list of twelve cards as 4-bit integers in
>         base 3 as strings, such as ["1022", "1122", ..., "1020"].
>     Returns:
>         (int) The number of sets in the hand.
>     Raises:
>         ValueError: if the list does not contain a valid Set hand, meaning
>             - there are not exactly 12 cards,
>             - the cards are not all unique,
>             - one or more cards does not have exactly 4 digits, or
>             - one or more cards has a character other than 0, 1, or 2.
>     """
>     pass
>
> def is_set(a, b, c):
>     """Determine if the cards a, b, and c constitute a set.
>
>     Parameters:
>         a, b, c (str): string representations of 4-bit integers in base 3.
>             For example, "1022", "1122", and "1020" (which is not a set).
>     Returns:
>         True if a, b, and c form a set, meaning the ith digit of a, b,
>             and c are either the same or all different for i=1,2,3,4.
>         False if a, b, and c do not form a set.
>     """
>     pass
> ```
>
> Write unit tests for these functions, but *do not implement them yet*. Focus on *what* the functions should do rather than on *how* they will be implemented.
>
> (Hint: if three cards form a set, then the first digits of the cards are either all the same or all different. Then the sums of these digits can only be 0, 3, or 6. Thus a group of cards forms a set only if for each set of digits (first digits, second digits, etc.) the sum is a multiple of 3.)
>
> ¹ A 4-bit integer in base 3 contains four digits that are either 0, 1 or 2. For example, 0000 and 1201 are 4-bit integers in base 3, whereas 000 is not because it has only three digits, and 0123 is not because it contains the number 3.

The tests check the number of sets in a valid hand, each of the four `ValueError` conditions (with its message), and individual triples that are and aren't sets:

In [16]:
show(test_specs.test_count_sets, test_specs.test_is_set)

```python
def test_count_sets():
    hand1 = ["1022", "1122", "0100", "2021",
            "0010", "2201", "2111", "0020",
            "1102", "0200", "2110", "1020"]
    assert specs.count_sets(hand1) == 6
    hand2 = hand1.copy()
    hand2.remove("1020")
    with pytest.raises(ValueError) as excinfo_1:
        specs.count_sets(hand2)
    assert excinfo_1.value.args[0] == "there are not exactly 12 cards"
    hand3 = hand1.copy()
    hand3[11] = "2110"
    with pytest.raises(ValueError) as excinfo_2:
        specs.count_sets(hand3)
    assert excinfo_2.value.args[0] == "the cards are not all unique"
    hand4 = hand1.copy()
    hand4[11] = "10201"
    with pytest.raises(ValueError) as excinfo_3:
        specs.count_sets(hand4)
    assert excinfo_3.value.args[0] == "one or more cards does not have exactly 4 digits"
    hand5 = hand1.copy()
    hand5[11] = "1420"
    with pytest.raises(ValueError) as excinfo_4:
        specs.count_sets(hand5)
    assert excinfo_4.value.args[0] == "one or more cards has a character other than 0, 1, or 2"

def test_is_set():
    hand1 = ["1022", "1122", "0100", "2021",
            "0010", "2201", "2111", "0020",
            "1102", "0200", "2110", "1020"]
    assert specs.is_set(hand1[0], hand1[3], hand1[7]) == True
    assert specs.is_set(hand1[3], hand1[6], hand1[5]) == True
    assert specs.is_set(hand1[11], hand1[10], hand1[9]) == True
    assert specs.is_set(hand1[1], hand1[6], hand1[2]) == True
    assert specs.is_set(hand1[0], hand1[6], hand1[9]) == True
    assert specs.is_set(hand1[1], hand1[4], hand1[5]) == True
    assert specs.is_set(hand1[1], hand1[2], hand1[3]) == False
```

## Problem 6: Implementing Set

> **Problem 6.** After you have written unit tests for the functions in Problem 5, implement the actual functions. If needed, add additional test cases to get full coverage.
>
> (Hint: The `combinations()` function from the standard library module `itertools` may be useful in implementing `count_sets()`.)

`count_sets()` validates the hand, then counts the 3-card combinations that `is_set()` accepts; `is_set()` checks that each digit position is all the same or all different.

In [17]:
show(specs.count_sets, specs.is_set)

```python
def count_sets(cards):
    """Return the number of sets in the provided Set hand.

    Parameters:
        cards (list(str)) a list of twelve cards as 4-bit integers in
        base 3 as strings, such as ["1022", "1122", ..., "1020"].
    Returns:
        (int) The number of sets in the hand.
    Raises:
        ValueError: if the list does not contain a valid Set hand, meaning
            - there are not exactly 12 cards,
            - the cards are not all unique,
            - one or more cards does not have exactly 4 digits, or
            - one or more cards has a character other than 0, 1, or 2.
    """
    from itertools import combinations

    if len(cards) != 12:
        raise ValueError("there are not exactly 12 cards")
    for i in cards:
        if len(i) != 4:
            raise ValueError("one or more cards does not have exactly 4 digits")
        for k in i:
            if k != '0' and k != '1' and k != '2':
                raise ValueError("one or more cards has a character other than 0, 1, or 2")
        for j in cards[cards.index(i)+1:]:
            if i == j:
                raise ValueError("the cards are not all unique")
    s = list(combinations(cards, 3))
    count = 0
    for i in s:
        count += int(is_set(i[0], i[1], i[2]))
    return count

def is_set(a, b, c):
    """Determine if the cards a, b, and c constitute a set.

    Parameters:
        a, b, c (str): string representations of 4-bit integers in base 3.
            For example, "1022", "1122", and "1020" (which is not a set).
    Returns:
        True if a, b, and c form a set, meaning the ith digit of a, b,
            and c are either the same or all different for i=1,2,3,4.
        False if a, b, and c do not form a set.
    """
    for i,j,k in zip(a, b, c):
        if i == j and i == k and j == k:
            continue
        elif i != j and i != k and j != k:
            continue
        else:
            return False
    return True
```

In [18]:
hand1 = ["1022", "1122", "0100", "2021",
         "0010", "2201", "2111", "0020",
         "1102", "0210", "2110", "1020"]
print(specs.count_sets(hand1))                      # The spec's hand has six sets.
print(specs.is_set("1022", "1122", "1020"))         # The docstring's example of a non-set.

6
False


### The full test suite

In [19]:
!{sys.executable} -m pytest -v --no-header -p no:cacheprovider test_specs.py

============================= test session starts ==============================
collecting ... 

collected 15 items                                                             

test_specs.py::test_add PASSED                                           [  6%]
test_specs.py::test_divide PASSED                                        [ 13%]
test_specs.py::test_smallest_factor PASSED                               [ 20%]
test_specs.py::test_month_length PASSED                                  [ 26%]
test_specs.py::test_operate PASSED                                       [ 33%]
test_specs.py::test_fraction_init PASSED                                 [ 40%]
test_specs.py::test_fraction_str PASSED                                  [ 46%]
test_specs.py::test_fraction_float PASSED                                [ 53%]
test_specs.py::test_fraction_eq PASSED                                   [ 60%]
test_specs.py::test_fraction_add PASSED                                  [ 66%]
test_specs.py::test_fraction_sub PASSED                                  [ 73%]
test_specs.py::test_fraction_mul PASSED

## Additional Material

### The Python Debugger

Python's built-in debugger `pdb` helps find mistakes during execution, in a terminal or a Jupyter Notebook. `pdb.set_trace()` sets a *break point* where the program pauses; then these commands control it:

| Command | Description |
|---|---|
| `n` | next: executes the next line |
| `p <var>` | print: display the value of the specified variable. |
| `c` | continue: stop debugging and run the program normally to the end. |
| `q` | quit: terminate the program. |
| `l` | list: show several lines of code around the current line. |
| `r` | return: return to the end of a subroutine. |
| `<Enter>` | Execute the most recent command again. |

For example, in a long loop where a variable changes unpredictably, `p i` checks its value at any step, and it can even be changed mid-program:

```python
# pdb_example.py
import pdb
from random import randint

i = 0
pdb.set_trace()     # Set a break point.
while i < 1000000000:
    i += randint(1, 10)
print("DONE")
```

```text
$ python pdb_example.py
> /Users/Student/pdb_example.py(7)<module>()
-> while i < 1000000000:
(Pdb) n                 # Execute a few lines.
> /Users/Student/pdb_example.py(8)<module>()
-> i += randint(1, 10)
(Pdb) n
> /Users/Student/pdb_example.py(7)<module>()
-> while i < 1000000000:
(Pdb) p i               # Check the value of i.
8
(Pdb) i = 999999999     # Change the value of i.
(Pdb) c                 # Continue the program.
DONE
```

See https://docs.python.org/3/library/pdb.html for documentation and examples.

### Other Testing Suites

Other frameworks share the same basic structure as `pytest` with different setup, syntax, and features: the standard library's `unittest` (https://docs.python.org/3/library/unittest.html) and `doctest` (https://docs.python.org/3/library/doctest.html), or the third-party `nose` module (https://nose.readthedocs.io/en/latest/). For a much larger list, see https://wiki.python.org/moin/PythonTestingToolsTaxonomy.

### The Fractions Module

The standard library's `fractions` module (https://docs.python.org/3/library/fractions.html) has a `Fraction` class similar to the one in this lab, but a little more robust: its constructor accepts floats, decimals, integers, and strings. See also the `decimal` module (https://docs.python.org/3/library/decimal.html) for decimal arithmetic.

In [20]:
import fractions
fractions.Fraction(8, 12), fractions.Fraction(0.75), fractions.Fraction("3/15"), fractions.Fraction(1, 2) + fractions.Fraction(1, 3)

(Fraction(2, 3), Fraction(3, 4), Fraction(1, 5), Fraction(5, 6))